In [ ]:
#| default_exp tasks

# tasks

> `classify(folder, model)`. The whole library in one line per job.

Each function here loads a model (once, cached), runs it over whatever you point at, and returns
`Preds`. They exist so that the common case is one call and the uncommon case is still `Model`.

In [ ]:
#| export
from __future__ import annotations
from pathlib import Path

import numpy as np
from fastcore.all import AttrDict, L

from anya.core import Model, Pred, Preds, arrange, items, load_model
from anya.vision import similarity

In [ ]:
#| hide
from fastcore.test import *
from tempfile import mkdtemp
from PIL import Image
FIX = Path('fixtures')

## One line per task

`model` is a path, a hub repo id, an alias, or a `Model` you already built. The task functions do
not guess a default model: a classifier for Australian birds and one for chest X-rays are both
"classify", and picking one for you would be picking wrong.

In [ ]:
#| export
def as_model(model, **kw) -> Model:
    'A loaded `Model` from a path, a repo id, an alias, or a `Model` (returned as is).'
    if isinstance(model, Model): return model
    if model is None: raise ValueError(
        'anya has no default model: pass a path, a hub repo id, or an alias. '
        'anya.hub.find_models("what it should do") searches for one.')
    return load_model(model, **kw)

def run(x,                  # a file, a folder, a glob, a list, or pixels
        model,              # a path, repo id, alias, or Model
        task:str=None,      # override the task the model's outputs imply
        **kw                # model keywords (labels=, size=, norm=…) and call keywords (topk=, conf=…)
       ):
    'Run any model over anything: one item gives a `Pred`, a folder gives `Preds`.'
    call = {k: kw.pop(k) for k in ('topk', 'conf', 'iou', 'bs', 'on_error', 'types', 'exclude') if k in kw}
    m = as_model(model, task=task, **kw)
    if isinstance(x, (str, Path)) and Path(str(x)).suffix.lower() in {'.mp4','.mov','.avi','.mkv','.webm','.m4v'}:
        return m.predict_video(x, **{k: v for k, v in call.items() if k in ('topk','conf','iou')})
    return m(x, **call)

In [ ]:
#| export
def classify(x, model, topk:int=5, **kw):
    'What is in this picture? Ranked labels per item.'
    return run(x, model, task=kw.pop('task', 'classify'), topk=topk, **kw)

def detect(x, model, conf:float=0.25, iou:float=0.45, **kw):
    'What objects are in this picture, and where? Boxes per item.'
    return run(x, model, task=kw.pop('task', 'detect'), conf=conf, iou=iou, **kw)

def segment(x, model, **kw):
    'Which pixels are what? A label map plus the share of the picture each class covers.'
    return run(x, model, task=kw.pop('task', 'segment'), **kw)

def embed(x, model, **kw):
    'A unit-length feature vector per item, for comparing pictures to each other.'
    return run(x, model, task=kw.pop('task', 'embed'), **kw)

In [ ]:
#| hide
_m = Model(FIX/'tiny_cls.onnx', labels=['red','green','blue','none'])
green = np.zeros((16,16,3), np.uint8); green[...,1] = 255
test_eq(classify(green, _m).label, 'green')
test_eq(len(classify(green, _m, topk=2).preds), 2)
test_eq(detect(np.zeros((50,100,3), np.uint8), Model(FIX/'tiny_det.onnx', labels=['a','b','c'])).objects[0]['label'], 'b')
test_eq(segment(green, Model(FIX/'tiny_seg.onnx', labels=['red','green','blue'])).label, 'green')
test_eq(embed(green, Model(FIX/'tiny_emb.onnx')).vec.shape, (128,))
test_fail(lambda: classify(green, None), contains='no default model')

## Sorting a folder

The job the whole library is for: run a classifier over a folder and put the files where the labels
say. `dry_run=True` by default, so the first call answers "what would you do" and the second one
does it.

In [ ]:
#| export
def sort_images(folder,                  # folder of pictures to sort
                model,                   # classifier: a path, repo id, alias, or Model
                dest=None,               # where the sorted tree goes; defaults to `folder`/sorted
                min_score:float=0.5,     # anything less confident goes to `other`
                how:str='copy',          # 'copy', 'move', or 'link'
                other:str='unsorted',
                dry_run:bool=True,       # show the plan, change nothing
                **kw
               ) -> AttrDict:
    'Classify a folder and file every picture under `dest/<label>/`.'
    dest = Path(dest or Path(folder)/'sorted')
    ps = classify(folder, model, topk=1, exclude=dest, **kw)     # so a second run does not read the first one's output
    r = arrange(ps, dest, how=how, min_score=min_score, other=other, dry_run=dry_run)
    return AttrDict(r, counts=ps.counts(), failed=len(ps.failed), preds=ps)

def summarize(preds:Preds, top:int=10) -> AttrDict:
    'What a run found, small enough to read or to hand to a model.'
    c = preds.counts()
    return AttrDict(n=len(preds), ok=len(preds.ok), failed=len(preds.failed),
                    labels=len(c), counts=dict(list(c.items())[:top]),
                    mean_score=round(float(np.mean([p.score for p in preds.ok if p.score is not None] or [0])), 4),
                    errors=[p['error'] for p in preds.failed[:3]])

In [ ]:
#| hide
_d = Path(mkdtemp())
for i, c in enumerate(['red','green','blue']):
    a = np.zeros((16,16,3), np.uint8); a[...,i] = 255
    Image.fromarray(a).save(_d/f'{c}.png')
(_d/'broken.png').write_bytes(b'nope')

_r = sort_images(_d, _m)
test_eq(_r.dry_run, True); test_eq((_d/'sorted').exists(), False)
test_eq(_r.counts, {'blue': 1, 'green': 1, 'red': 1}); test_eq(_r.failed, 1)
_r = sort_images(_d, _m, dry_run=False)
test_eq(sorted(p.name for p in (_d/'sorted').iterdir()), ['blue','green','red','unsorted'])
test_eq(sort_images(_d, _m).counts, {'blue': 1, 'green': 1, 'red': 1})   # the default dest is inside the
                                                                        # folder, and is skipped on a rerun
test_eq(summarize(_r.preds).failed, 1)
test_eq(summarize(_r.preds).labels, 3)

## Finding pictures by picture

An embedder plus a cosine is the cheapest useful search: no labels, no training, works on a folder
you assembled this morning.

In [ ]:
#| export
def find_similar(query,                  # one picture: a path or pixels
                 folder,                 # where to look
                 model,                  # an embedding model
                 n:int=10,               # how many to return
                 **kw
                ) -> L:
    'Rank the pictures in `folder` by cosine similarity to `query`.'
    m = as_model(model, task='embed', **kw)
    q, ps = m.predict(query), m.predict_all(folder)
    ok = ps.ok
    if not len(ok): return L()
    sims = similarity(q.vec, np.stack([p.vec for p in ok]))[0]
    order = np.argsort(-sims)[:n]
    return L(AttrDict(src=ok[int(i)]['src'], score=round(float(sims[i]), 6)) for i in order)

def index_folder(folder, model, **kw) -> AttrDict:
    'Embed every picture in `folder` once, for repeated comparisons.'
    m = as_model(model, task='embed', **kw)
    ps = m.predict_all(folder).ok
    return AttrDict(srcs=L(p['src'] for p in ps), vecs=np.stack([p.vec for p in ps]) if len(ps) else np.zeros((0, 1)),
                    model=m.name, n=len(ps))

In [ ]:
#| hide
_sd = Path(mkdtemp())                              # a folder of its own: _d now holds its own sorted copies
for i, c in enumerate(['red','green','blue']):
    a = np.zeros((16,16,3), np.uint8); a[...,i] = 255
    Image.fromarray(a).save(_sd/f'{c}.png')
_e = Model(FIX/'tiny_emb.onnx')
_s = find_similar(_sd/'green.png', _sd, _e, n=2)
test_eq(Path(_s[0]['src']).name, 'green.png')      # a picture is most like itself
test_close(_s[0]['score'], 1.0, eps=1e-5)
test_eq(index_folder(_sd, _e).vecs.shape, (3, 128))

## How fast

Worth a call before committing to 2000 files: it says images per second on this machine at this
batch size, which is what decides whether to wait or to go and do something else.

In [ ]:
#| export
def bench(model,                 # a path, repo id, alias, or Model
          size:tuple=(640, 640), # size of the fake pictures to push through
          n:int=20,              # how many
          bs:int=None,
          **kw
         ) -> AttrDict:
    'Images per second for one model on this machine, measured on random pictures.'
    import time
    m = as_model(model, **kw)
    xs = [np.random.randint(0, 255, (*size, 3), dtype=np.uint8) for _ in range(n)]
    m.predict(xs[0])                                   # the first call pays for allocation
    t0 = time.perf_counter(); m.predict_all(xs, bs=bs); dt = time.perf_counter() - t0
    return AttrDict(model=m.name, runtime=m.runtime, task=m.task, n=n, bs=bs or m.max_bs,
                    seconds=round(dt, 3), per_second=round(n/dt, 1), ms_each=round(1000*dt/n, 2))

In [ ]:
#| hide
_b = bench(_m, size=(32, 32), n=4)
test_eq(_b.n, 4); test_eq(_b.per_second > 0, True); test_eq(_b.runtime, 'onnx')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()